In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from ingest import load_pdfs, chunk_documents, build_index
from query import load_index, retrieve

pages = load_pdfs(config.DATA_DIR)
chunks = chunk_documents(pages)
vectordb = build_index(chunks)
print(f"\nIndex ready: {len(chunks)} chunks from {len(pages)} pages.")

Loading Guideline for the pharmacological treatment of hypertension in adults.pdf ...


  -> 61 pages loaded
Loading WHO_Hypertension_Guideline_2021.pdf ...
  -> 13 pages loaded


/Users/alikhalidalikhalid/Downloads/Day One/Task/ingest.py:36: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODEL)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 217 chunks using 'local' provider ...


Done. Index saved to /Users/alikhalidalikhalid/Downloads/Day One/Task/chroma_db/

Index ready: 217 chunks from 74 pages.


In [2]:
question = "What is the target blood pressure for a patient with cardiovascular disease?"

for k in [1, 3, 8]:
    results = retrieve(vectordb, question, k=k)
    print(f"--- k={k} ---")
    for doc, score in results:
        print(f"  score={score:.3f}  page {doc.metadata.get('page_number')}: "
              f"{doc.page_content[:70].strip()}...")
    print()

--- k=1 ---
  score=0.481  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...

--- k=3 ---
  score=0.481  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...
  score=0.481  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...
  score=0.471  page 5: WHO Guideline for the Pharmacological Treatment of Hypertension in Adu...

--- k=8 ---
  score=0.481  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...
  score=0.481  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...
  score=0.471  page 5: WHO Guideline for the Pharmacological Treatment of Hypertension in Adu...
  score=0.471  page 5: WHO Guideline for the Pharmacological Treatment of Hypertension in Adu...
  score=0.467  page 9: hypertension (those with high CVD risk, diabetes mellitus, chronic kid...
  score=0.467  page 9: hypertension (those with high CVD risk, diabetes mellitus, chronic

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ingest import get_embedding_function
from langchain_chroma import Chroma

test_queries = [
    "What blood pressure threshold should trigger starting medication?",
    "What are the three recommended first-line drug classes?",
    "Can nurses or pharmacists prescribe antihypertensive treatment?",
    "Does hypertension increase the risk of severe COVID-19?",
    "Is monotherapy or combination therapy preferred?",
]

configurations = [
    {"name": "Small (200/0)",    "chunk_size": 200,  "chunk_overlap": 0},
    {"name": "Balanced (400/50)", "chunk_size": 400,  "chunk_overlap": 50},
    {"name": "Large (600/100)",  "chunk_size": 600,  "chunk_overlap": 100},
]

embed_fn = get_embedding_function()
experiment_results = []

for cfg in configurations:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"] * 4,
        chunk_overlap=cfg["chunk_overlap"] * 4,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    test_chunks = splitter.split_documents(pages)
    test_db = Chroma.from_documents(
        documents=test_chunks, embedding=embed_fn,
        collection_name=f"experiment_{cfg['chunk_size']}",
    )

    avg_score = 0
    for q in test_queries:
        results = test_db.similarity_search_with_relevance_scores(q, k=3)
        avg_score += sum(s for _, s in results) / len(results)
    avg_score /= len(test_queries)

    experiment_results.append({"config": cfg["name"], "n_chunks": len(test_chunks), "avg_top3_score": avg_score})
    print(f"{cfg['name']:<20} chunks={len(test_chunks):>4}   avg top-3 relevance={avg_score:.3f}")

print("\n" + "=" * 65)
print("ABLATION SUMMARY")
print("=" * 65)
sorted_results = sorted(experiment_results, key=lambda x: x['avg_top3_score'], reverse=True)
for rank, r in enumerate(sorted_results, 1):
    marker = " <-- BEST" if rank == 1 else ""
    print(f"{r['config']:<22} {r['n_chunks']:>8} chunks  {r['avg_top3_score']:>8.3f}{marker}")
print("=" * 65)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Small (200/0)        chunks= 282   avg top-3 relevance=0.470


Balanced (400/50)    chunks= 163   avg top-3 relevance=0.413


Large (600/100)      chunks= 119   avg top-3 relevance=0.414

ABLATION SUMMARY
Small (200/0)               282 chunks     0.470 <-- BEST
Large (600/100)             119 chunks     0.414
Balanced (400/50)           163 chunks     0.413


In [4]:
import csv

test_set = []
with open("../eval/Day2_Evaluation_Test_Set.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        test_set.append(row)

print(f"Loaded {len(test_set)} test questions.\n")
for i, row in enumerate(test_set, 1):
    marker = " [OUT-OF-SCOPE]" if "Not covered" in row["Expected Source (Document / Section / Page)"] else ""
    print(f"  Q{i:>2}. {row['Question']}{marker}")

Loaded 15 test questions.

  Q 1. What blood pressure threshold should trigger starting medication for hypertension?
  Q 2. What are the three recommended first-line drug classes for treating hypertension?
  Q 3. What is the target blood pressure for a patient with existing cardiovascular disease?
  Q 4. How often should a patient be followed up after starting antihypertensive treatment?
  Q 5. Can nurses or pharmacists prescribe antihypertensive treatment?
  Q 6. Which antihypertensive medications are contraindicated during pregnancy?
  Q 7. What is the recommended starting dose in the single-pill combination strategy?
  Q 8. Does hypertension increase the risk of severe COVID-19?
  Q 9. What laboratory tests should be performed before initiating antihypertensive treatment?
  Q10. Should patients continue ACE inhibitors or ARBs during COVID-19 infection?
  Q11. What is the recommended approach for managing hypertension in disaster or humanitarian settings?
  Q12. What drug-specific pr

In [5]:
import re

def page_matches_expected(retrieved_page, expected_text):
    m = re.search(r"Page (\d+)", expected_text)
    if not m:
        return None
    return retrieved_page == int(m.group(1))

k = 3
precisions = []
print(f"{'#':<4} {'Question':<60} {'P@3':<8} {'Status'}")
print("=" * 95)

for i, row in enumerate(test_set, 1):
    expected = row["Expected Source (Document / Section / Page)"]
    if "Not covered" in expected:
        print(f"{i:<4} {row['Question'][:58]:<60} {'N/A':<8} Out-of-scope")
        continue

    results = retrieve(vectordb, row["Question"], k=k)
    hits = sum(
        1 for doc, _ in results
        if page_matches_expected(doc.metadata.get("page_number"), expected)
    )
    precision = hits / k
    precisions.append(precision)
    status = "✓ HIT" if precision >= 1.0 else "✗ MISS"
    print(f"{i:<4} {row['Question'][:58]:<60} {precision:<8.2f} {status}")

avg_precision = sum(precisions) / len(precisions)
perfect = sum(1 for p in precisions if p >= 1.0)
print("=" * 95)
print(f"\nAverage Precision@{k}: {avg_precision:.2f}")
print(f"Perfect hits: {perfect}/{len(precisions)} questions")
print(f"Scored questions: {len(precisions)} (1 out-of-scope excluded)")

#    Question                                                     P@3      Status
1    What blood pressure threshold should trigger starting medi   0.67     ✗ MISS
2    What are the three recommended first-line drug classes for   0.00     ✗ MISS
3    What is the target blood pressure for a patient with exist   0.00     ✗ MISS


4    How often should a patient be followed up after starting a   0.67     ✗ MISS
5    Can nurses or pharmacists prescribe antihypertensive treat   0.00     ✗ MISS
6    Which antihypertensive medications are contraindicated dur   1.00     ✓ HIT
7    What is the recommended starting dose in the single-pill c   0.67     ✗ MISS
8    Does hypertension increase the risk of severe COVID-19?      0.00     ✗ MISS
9    What laboratory tests should be performed before initiatin   0.00     ✗ MISS
10   Should patients continue ACE inhibitors or ARBs during COV   0.67     ✗ MISS
11   What is the recommended approach for managing hypertension   0.67     ✗ MISS
12   What drug-specific protocols does the guideline provide fo   0.00     ✗ MISS
13   What percentage of pregnancy-related deaths are attributed   0.00     ✗ MISS
14   Is monotherapy or combination therapy preferred as initial   1.00     ✓ HIT
15   What screening interval does this guideline recommend for    N/A      Out-of-scope

Average Pre

In [6]:
print("DETAILED RETRIEVAL REPORT")
print("=" * 95)

for i, row in enumerate(test_set, 1):
    expected = row["Expected Source (Document / Section / Page)"]
    if "Not covered" in expected:
        continue
    
    print(f"\nQ{i}. {row['Question']}")
    print(f"    Expected: {expected}")
    results = retrieve(vectordb, row["Question"], k=3)
    for j, (doc, score) in enumerate(results, 1):
        page = doc.metadata.get('page_number')
        match = page_matches_expected(page, expected)
        status = "✓" if match else "✗"
        print(f"    [{j}] score={score:.3f} | page {page} | {status} | {doc.page_content[:60].strip()}...")
    print("-" * 95)

DETAILED RETRIEVAL REPORT

Q1. What blood pressure threshold should trigger starting medication for hypertension?
    Expected: Guideline for the pharmacological treatment of hypertension in adults / 3.1 Blood pressure threshold / Page 19
    [1] score=0.633 | page 19 | ✓ | 3 Recommendations
3.1 Blood pressure threshold for initiatio...
    [2] score=0.633 | page 19 | ✓ | 3 Recommendations
3.1 Blood pressure threshold for initiatio...
    [3] score=0.572 | page 2 | ✗ | Hypertension - or elevated blood pressure - is a serious med...
-----------------------------------------------------------------------------------------------

Q2. What are the three recommended first-line drug classes for treating hypertension?
    Expected: Guideline for the pharmacological treatment of hypertension in adults / 3.4 Drug classes / Page 24
    [1] score=0.630 | page 10 | ✗ | 4. RECOMMENDATION ON DRUG CLASSES TO BE USED AS FIRST-LINE A...
    [2] score=0.630 | page 10 | ✗ | 4. RECOMMENDATION ON DRUG CLAS

    [1] score=0.491 | page 30 | ✓ | For patients who were initiated on treatment, those who wait...
    [2] score=0.491 | page 30 | ✓ | For patients who were initiated on treatment, those who wait...
    [3] score=0.487 | page 9 | ✗ | WHO recommends pharmacological antihypertensive treatment of...
-----------------------------------------------------------------------------------------------

Q5. Can nurses or pharmacists prescribe antihypertensive treatment?
    Expected: Guideline for the pharmacological treatment of hypertension in adults / 3.8 Task sharing / Page 31
    [1] score=0.497 | page 9 | ✗ | WHO recommends pharmacological antihypertensive treatment of...
    [2] score=0.497 | page 9 | ✗ | WHO recommends pharmacological antihypertensive treatment of...
    [3] score=0.492 | page 25 | ✗ | persistence), as an initial treatment. Antihypertensive medi...
-----------------------------------------------------------------------------------------------

Q6. Which antihypertensive m